# 06 - K-Means Clustering & Model Evaluation (Phân cụm K-Means)

**Bám sát syllabus CHAPTER 5:** Machine Learning (Unsupervised Learning - Clustering).

**Mục tiêu đề tài (Q3):** *"Có thể phân khách hàng thành các nhóm có hành vi tương đồng hay không?"*

**Nguyên tắc thực hiện:**
1. **Input:** Bảng RFM đã qua tiền xử lý. **KHÔNG** dùng `CustomerID` làm đặc trưng gom cụm.
2. **Tiền xử lý:** Bắt buộc áp dụng `Log-transform: ln(x)` (giảm độ lệch) + `StandardScaler` (chuẩn hóa thang đo).
3. **Đánh giá khách quan:** Thử nghiệm dải $K = 2..7$, tính toán quán tính `Inertia (WCSS)` và hệ số `Silhouette Score` trước khi đưa ra quyết định chọn K.

## 0. Chuẩn bị thư viện & Nạp module

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.modeling.kmeans import (
    preprocess_rfm_for_clustering,
    evaluate_kmeans_clusters,
    plot_elbow_and_silhouette,
)

pd.options.display.float_format = "{:.4f}".format

rfm_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "rfm_data.csv")
print(f"Nạp thành công {len(rfm_df):,} khách hàng từ rfm_data.csv")

## 1. Tiền xử lý dữ liệu RFM (Log-Transform + StandardScaler)

**Lý do:**
- Frequency và Monetary có độ lệch phải rất lớn (Skewness > 12).
- Thang đo giữa các biến chênh lệch hàng nghìn lần (Recency tính bằng ngày, Monetary tính bằng hàng trăm nghìn bảng).
- Thuật toán K-Means sử dụng khoảng cách Euclid, bắt buộc phải đưa dữ liệu về phân phối tiệm cận chuẩn và cùng độ biến thiên.

In [ ]:
X_scaled, rfm_log, scaler = preprocess_rfm_for_clustering(rfm_df)

print("\nMẫu dữ liệu sau khi chuẩn hóa (5 dòng đầu):")
pd.DataFrame(X_scaled, columns=["Recency_scaled", "Frequency_scaled", "Monetary_scaled"]).head()

## 2. Thử nghiệm thuật toán K-Means trên dải K = 2..7

Tính toán 2 chỉ số học thuật:
1. **Inertia (WCSS):** Tổng bình phương khoảng cách từ các điểm dữ liệu đến tâm cụm tương ứng.
2. **Silhouette Score:** Đo lường mức độ tách biệt giữa các cụm (từ -1 đến 1, càng cao càng tốt).

In [ ]:
eval_df = evaluate_kmeans_clusters(X_scaled, k_range=range(2, 8))
eval_df

## 3. Trực quan hóa Phương pháp Elbow và Hệ số Silhouette

In [ ]:
fig = plot_elbow_and_silhouette(eval_df)
plt.show()

## 4. Phân tích Số liệu để Lựa chọn K Tối ưu

Dựa trên kết quả thực nghiệm:
| Số cụm K | Inertia (WCSS) | Silhouette Score | Nhận định phương pháp luận |
|:---:|:---:|:---:|:---|
| **K = 2** | 6,357.38 | **0.4312** | Silhouette cao nhất, nhưng chỉ phân đôi khách hàng (tương tác cao vs thấp), quá thô cho chiến lược tiếp thị thực tế. |
| **K = 3** | 4,805.89 | **0.3400** | Điểm gãy quán tính (Elbow) rõ ràng nhất, chia thành 3 phân khúc kinh điển (Khách rời bỏ, Khách tiềm năng, Khách trung thành). |
| **K = 4** | 3,915.39 | **0.3354** | Điểm uốn tiếp theo, Silhouette tương đương K=3 (chỉ chênh 0.004), cho phép tách thêm nhóm Khách hàng chi tiêu lớn (VIP). |
| **K = 5** | 3,356.04 | 0.3013 | Silhouette giảm sút rõ rệt, cụm bắt đầu bị phân mảnh nhỏ. |
| **K >= 6**| < 2,900 | ~0.309 | Phức tạp hóa phân khúc, khó khăn cho việc triển khai hành động kinh doanh. |

> 💡 **Đề xuất lựa chọn:**
> - **K = 3:** Nếu doanh nghiệp ưu tiên mô hình phân khúc tinh gọn, dễ quản lý.
> - **K = 4 (Khuyên dùng):** Nếu doanh nghiệp muốn tách bạch rõ ràng giữa nhóm khách hàng mua sắm thường xuyên với nhóm khách hàng VIP có giá trị tiền tệ đột biến để chăm sóc đặc biệt.